# Práctica: Embeddings y Vector Database (Azure AI Search)

> **Parte 2: Búsquedas prácticas en Python**

Este notebook ejecuta búsquedas reales sobre el índice creado en Azure AI Search: Vector Search, Hybrid Search, Semantic Search y Semantic Hybrid Search.

---

## Requisitos
- Tener el archivo `.env` configurado con tus claves y endpoints de Azure.
- Haber creado el índice y cargado los documentos siguiendo la Parte 1.
markdown
markdown
![](imagen13-extra-parte-1.png)

markdown
markdown
### Cómo se añadió el scoring profile en el índice
El scoring profile `boost_title` fue añadido manualmente desde el Portal de Azure, editando la definición del índice y configurando el campo `title` con un peso alto.


Demostrar el funcionamiento de los principales tipos de búsqueda sobre un índice vectorial en Azure AI Search, mostrando resultados claros y explicaciones para cada caso.## Objetivo---Puedes ver el proceso en la imagen adjunta a la entrega: **imagen13-extra-parte-1**.






Demostrar el funcionamiento de los principales tipos de búsqueda sobre un índice vectorial en Azure AI Search, mostrando resultados claros y explicaciones para cada caso.Esto garantiza que el código del notebook funcione correctamente y que el ranking personalizado esté activo en el índice.Demostrar el funcionamiento de los principales tipos de búsqueda sobre un índice vectorial en Azure AI Search, mostrando resultados claros y explicaciones para cada caso.## Objetivo---Puedes ver el proceso en la imagen adjunta a la entrega: **imagen13-extra-parte-1**.Demostrar el funcionamiento de los principales tipos de búsqueda sobre un índice vectorial en Azure AI Search, mostrando resultados claros y explicaciones para cada caso.

## 1. Cargar librerías y configuración
Carga de variables de entorno y librerías necesarias.

In [53]:
import os
from dotenv import load_dotenv
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
import openai
import pandas as pd

# Load environment variables
load_dotenv()

AZURE_SEARCH_ENDPOINT = os.getenv('AZURE_SEARCH_ENDPOINT')
AZURE_SEARCH_KEY = os.getenv('AZURE_SEARCH_KEY')
AZURE_SEARCH_INDEX = os.getenv('AZURE_SEARCH_INDEX')
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_KEY = os.getenv('AZURE_OPENAI_KEY')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION')
AZURE_SEARCH_SEMANTIC_CONFIG = os.getenv('AZURE_SEARCH_SEMANTIC_CONFIG')

# Initialize SearchClient
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX,
    credential=AzureKeyCredential(AZURE_SEARCH_KEY)
)

# Configure OpenAI
openai.api_type = 'azure'
openai.api_base = AZURE_OPENAI_ENDPOINT
openai.api_key = AZURE_OPENAI_KEY
openai.api_version = AZURE_OPENAI_API_VERSION

## 2. Utility Functions
Las siguientes funciones permiten obtener embeddings y mostrar los resultados de búsqueda en una tabla de forma clara. El código está en inglés para facilitar la comprensión internacional y la integración con otros proyectos.

In [54]:
# --- KEY FUNCTIONS FOR ALL SEARCHES ---

def get_embedding(text):
    """Get the embedding of a query using Azure OpenAI."""
    import openai
    response = openai.Embedding.create(
        input=text,
        engine=AZURE_OPENAI_EMBEDDING_DEPLOYMENT
)
    return response['data'][0]['embedding']

def show_results(results):
    """Show results in a table or indicate if there are no results."""
    if not results:
        print("No results found.")
        return
    import pandas as pd
    df = pd.DataFrame([
        {
            'score': r.get('@search.score', ''),
            'title': r.get('title', ''),
            'chunk': r.get('chunk', '')[:120] + '...'
        } for r in results
] )
    display(df)

## 3. Vector Search
Busca por similitud semántica usando embeddings.

**Explicación:**
La búsqueda vectorial utiliza embeddings generados por Azure OpenAI para encontrar los fragmentos más similares semánticamente a la consulta, aunque no compartan palabras exactas. Ideal para preguntas abiertas o conceptos complejos.

**Ejemplo de consulta:**
- ¿Qué es el algoritmo HNSW?

**Resultado:**
Se muestran los 5 fragmentos más relevantes según la similitud de embeddings. El código está en inglés para facilitar la integración y la colaboración internacional.

In [55]:
query = 'What is the HNSW algorithm?'
vector = get_embedding(query)
results = search_client.search(
    vector_queries=[{
        'vector': vector,
        'k': 5,
        'fields': 'text_vector',
        'kind': 'vector'
    }],
    top=5
)
show_results(list(results))

,score,title,chunk
0,0.724979,04_hnsw_algorithm.txt,TECHNICAL DOCUMENTATION REGARDING HNSW_ALGORIT...
1,0.656825,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
2,0.655466,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
3,0.655466,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
4,0.655466,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...


## 4. Hybrid Search
Combina búsqueda por palabra clave y vector.

**Explicación:**
La búsqueda híbrida suma la fuerza de la búsqueda por palabras clave (BM25) y la búsqueda vectorial. Así, encuentra resultados que coinciden tanto léxica como semánticamente con la consulta.

**Ejemplo de consulta:**
- vectorización eficiente

**Resultado:**
Se muestran los 5 fragmentos más relevantes considerando ambos enfoques. El código está en inglés para facilitar la integración y la colaboración internacional.

In [56]:
query = 'efficient vectorization'
vector = get_embedding(query)
results = search_client.search(
    search_text=query,
    vector_queries=[{
        'vector': vector,
        'k': 5,
        'fields': 'text_vector',
        'kind': 'vector'
    }],
    top=5
)
show_results(list(results))

,score,title,chunk
0,0.016667,02_vector_embeddings.txt,TECHNICAL DOCUMENTATION REGARDING VECTOR_EMBED...
1,0.016393,02_vector_embeddings.txt,vectors within a multidimensional latent space...
2,0.016129,02_vector_embeddings.txt,vectors within a multidimensional latent space...
3,0.015873,02_vector_embeddings.txt,vectors within a multidimensional latent space...
4,0.015625,02_vector_embeddings.txt,vectors within a multidimensional latent space...


## 5. Semantic Search
Búsqueda semántica usando el semantic ranker.

**Explicación:**
La búsqueda semántica utiliza el semantic ranker de Azure para entender la intención de la consulta y reordenar los resultados según su relevancia semántica, no solo por coincidencia de palabras.

**Ejemplo de consulta:**
- ¿Cómo se asegura la seguridad en Azure?

**Resultado:**
Se muestran los 5 fragmentos más relevantes según el análisis semántico del contenido. El código está en inglés para facilitar la integración y la colaboración internacional.

In [57]:
query = 'How is security ensured in Azure?'
results = search_client.search(
    search_text=query,
    query_type='semantic',
    semantic_configuration_name=AZURE_SEARCH_SEMANTIC_CONFIG,
    top=5
)
show_results(list(results))

,score,title,chunk
0,1.854090,08_cloud_security.txt,TECHNICAL DOCUMENTATION REGARDING CLOUD_SECURI...
1,6.173190,08_cloud_security.txt,keys in the source code. Using Managed Identit...
2,6.447996,08_cloud_security.txt,keys in the source code. Using Managed Identit...
3,6.202750,08_cloud_security.txt,keys in the source code. Using Managed Identit...
4,5.133820,08_cloud_security.txt,keys in the source code. Using Managed Identit...


## 6. Semantic Hybrid Search
Combina semantic ranker y vector search.

**Explicación:**
La búsqueda híbrida semántica combina el poder del semantic ranker y la búsqueda vectorial, logrando resultados aún más precisos y relevantes para consultas complejas.

**Ejemplo de consulta:**
- procesamiento por lotes en Azure

**Resultado:**
Se muestran los 5 fragmentos más relevantes considerando tanto la semántica como la similitud vectorial.

In [58]:
# Semantic Hybrid Search: combines semantic ranker and vector search

query = 'batch processing in Azure'

vector = get_embedding(query)

results = search_client.search(

    search_text=query,

    query_type='semantic',

    semantic_configuration_name=AZURE_SEARCH_SEMANTIC_CONFIG,

    vector_queries=[{

        'vector': vector,

        'k': 5,

        'fields': 'text_vector',

        'kind': 'vector'

    }],

    top=5

)

show_results(list(results))

,score,title,chunk
0,0.015152,09_batch_processing.txt,TECHNICAL DOCUMENTATION REGARDING BATCH_PROCES...
1,0.032787,09_batch_processing.txt,Batch processing allows for handling massive d...
2,0.032292,09_batch_processing.txt,Batch processing allows for handling massive d...
3,0.031054,09_batch_processing.txt,Batch processing allows for handling massive d...
4,0.030579,09_batch_processing.txt,Batch processing allows for handling massive d...


## 7. Extra: Ranking personalizado con Scoring Profile



Esta sección extra demuestra cómo utilizar un scoring profile en Azure AI Search para personalizar el ranking de los resultados de búsqueda. Al potenciar el campo `title`, los documentos que coincidan en el título aparecerán más arriba en los resultados.



Comparamos los resultados de una búsqueda estándar con los de una búsqueda usando el scoring profile `boost_title`. Así puedes ver el impacto real de un ranking personalizado en la experiencia de búsqueda.



La siguiente celda de código muestra la comparativa.

### Cómo se añadió el scoring profile en el índice
El scoring profile `boost_title` fue añadido manualmente desde el Portal de Azure, editando la definición del índice y configurando el campo `title` con un peso alto.

Puedes ver el proceso en la imagen adjunta:

![](imagen13-extra-parte1.jpg)



In [59]:
# Comparison: standard search vs scoring profile

# We use a query that matches exactly the title field of the documents

query = '04_hnsw_algorithm.txt'



# Standard search (without scoring profile)

results_default = search_client.search(

    search_text=query,

    top=5

)

print('Results WITHOUT scoring profile:')

show_results(list(results_default))



# Search with scoring profile

results_boost = search_client.search(

    search_text=query,

    scoring_profile='boost_title',

    top=5

)

print('Results WITH scoring profile (boost_title):')

show_results(list(results_boost))

Results WITHOUT scoring profile:


,score,title,chunk
0,1.791759,04_hnsw_algorithm.txt,TECHNICAL DOCUMENTATION REGARDING HNSW_ALGORIT...
1,1.673976,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
2,1.540445,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
3,1.540445,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
4,1.386294,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...


Results WITH scoring profile (boost_title):


,score,title,chunk
0,5.375279,04_hnsw_algorithm.txt,TECHNICAL DOCUMENTATION REGARDING HNSW_ALGORIT...
1,5.021930,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
2,4.621335,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
3,4.621335,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...
4,4.158884,04_hnsw_algorithm.txt,for nearest neighbor searches with ultra-low l...


## Conclusión Final



Se ha demostrado cómo implementar y comparar diferentes tipos de búsqueda sobre un índice vectorial en Azure AI Search, incluyendo la personalización del ranking mediante un scoring profile (`boost_title`).



**Puntos clave:**

- El uso de scoring profiles permite adaptar el ranking a las necesidades del negocio, dando más peso a campos relevantes como el título.

- La comparativa muestra cómo el orden de los resultados puede cambiar significativamente, mejorando la relevancia para el usuario final.

- Azure AI Search ofrece flexibilidad para combinar búsquedas léxicas, semánticas y personalizadas, facilitando soluciones avanzadas de recuperación de información.


